# Notebook 00c — Advanced Clean Extraction

> **Stage:** 00c | **Furnace:** parameterised via papermill  
> **Predecessor:** `notebook_00b_clean_extraction.ipynb`  
> Implements `analysis_window_v2` — 3-layer filter using `decoke_air` ground truth, `cot_ctrl` setpoint stability, and PELT change-point detection.

In [0]:
%pip install ruptures

## 1. Parameters
Same as 00b. New parameters: `PELT_PENALTY`, `COT_STEP_THRESH`, `COT_STD_ADV_C`.

In [0]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, REPO_ROOT)

os.environ['DDF_CLUSTER_ID'] = '0612-154044-62gqpkte'

from olefins_ddf.io_events import get_spark
from olefins_ddf import catalog as cat, features as feat_mod
from olefins_ddf.runs import segment_runs, cracking_mask

# ── Core parameters (same as 00b) ────────────────────────────────────────
FURNACE       = '1HA'           # papermill injects this
START         = '2025-02-05'
END           = '2026-03-20'
BUCKET_MIN    = 30
DELTA_CATALOG = 'indorama_corporate_olefins_paas_azure_weu_dev_research.workspaces'
OUTPUT_DIR    = os.path.join(REPO_ROOT, 'output', FURNACE, '00c_clean_adv')

# ── 00b heuristic thresholds (kept for fallback + comparison) ────────────
FEED_SETTLED_FRAC = 0.95
COT_STD_THRESH_C  = 2.0        # rolling σ threshold (measured COT, 00b style)
COT_ROLL_STEPS    = 8
MIN_WARMUP_H      = 12.0
MIN_GAP_H         = 24

# ── 00c new parameters ───────────────────────────────────────────────────
DECOKE_AIR_THRESH = 0.0        # Nm³/h — any value > 0 = decoke in progress
COT_STEP_THRESH   = 0.5        # °C — step change in cot_ctrl setpoint = operator intervention
COT_STD_ADV_C     = 1.0        # °C — tighter rolling σ on cot_ctrl (setpoint, less noise)
PELT_PENALTY      = 10.0       # PELT penalty β — tune in [5, 20]; lower → more breakpoints
PELT_MIN_SIZE     = 12         # minimum segment length in rows (12 × 30min = 6h)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Furnace      : {FURNACE}')
print(f'Window       : {START} → {END}')
print(f'Cluster      : {os.environ["DDF_CLUSTER_ID"]}')
print(f'PELT penalty : {PELT_PENALTY}  (tune in [5, 20])')
print(f'Output dir   : {OUTPUT_DIR}')

## 2. Connect to Databricks & load catalog

In [0]:
spark   = get_spark(os.environ['DDF_CLUSTER_ID'])
catalog = cat.build_catalog(spark)

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {DELTA_CATALOG}')
print(f'Schema ready: {DELTA_CATALOG}')

tube_tags = catalog[catalog['role'] == 'tube_COT']
if 'furnace' in tube_tags.columns:
    tube_tags = tube_tags[tube_tags['furnace'] == FURNACE]
print(f'Catalog loaded : {len(catalog)} total tags')
print(f'tube_COT ({FURNACE}) : {len(tube_tags)} tags (expect 192)')

# Report available ground truth tags for this furnace
gt_roles = ['decoke_air', 'cot_ctrl']
for role in gt_roles:
    gt_tags = catalog[catalog['role'] == role]
    if 'furnace' in gt_tags.columns:
        gt_tags = gt_tags[gt_tags['furnace'] == FURNACE]
    print(f'{role:12s} tags ({FURNACE}): {len(gt_tags)}  — {list(gt_tags.index) if len(gt_tags) <= 8 else list(gt_tags.index[:4]) + ["..."]}')

## 3. Load feature matrix
Load from Delta (00b must have been run first to populate the cache).
If not found, builds from raw historian data (~5–10 min).

In [0]:
DELTA_TABLE_FEAT = f'{DELTA_CATALOG}.{FURNACE.lower()}_features'

try:
    print(f'Loading feature matrix from Delta: {DELTA_TABLE_FEAT}')
    feat_spark = spark.table(DELTA_TABLE_FEAT)
    feat = feat_spark.toPandas()
    if 'timestamp' in feat.columns:
        feat = feat.set_index('timestamp')
        feat.index = pd.to_datetime(feat.index)
        feat = feat.sort_index()
    elif 'ts' in feat.columns:
        feat = feat.set_index('ts')
        feat.index = pd.to_datetime(feat.index)
        feat = feat.sort_index()
    feat.index.name = 'timestamp'
    print(f'Loaded {len(feat):,} rows from Delta cache.')
except Exception as e:
    print(f'Delta cache not found ({e}). Building from Databricks (~5–10 min)...')
    feat, _ = feat_mod.build_feature_matrix(
        spark, catalog, FURNACE, start=START, end=END, bucket=BUCKET_MIN
    )
    sdf = spark.createDataFrame(feat.reset_index().rename(columns={'index': 'timestamp'}))
    (sdf.write
       .format('delta')
       .mode('overwrite')
       .option('overwriteSchema', 'true')
       .saveAsTable(DELTA_TABLE_FEAT))
    print(f'Saved to Delta: {DELTA_TABLE_FEAT}')

print(f'Feature matrix : {feat.shape[0]:,} rows × {feat.shape[1]} columns')
print(f'Time range     : {feat.index.min()} → {feat.index.max()}')

# Report which ground truth columns are actually present
decoke_air_cols = [c for c in feat.columns if c.startswith('decoke_air')]
cot_ctrl_cols   = [c for c in feat.columns if c == 'cot_ctrl']
print(f'\ndecoke_air columns in feat : {decoke_air_cols}')
print(f'cot_ctrl columns in feat   : {cot_ctrl_cols}')

## 4. Detect cracking runs
Same as 00b: uses HC feed drop to find run boundaries.
`decoke_air` is used inside each run's analysis window (Section 5), not here.
> Decoke does NOT cool the furnace — run detection uses feed drop, not temperature.

In [0]:
if not isinstance(feat.index, pd.DatetimeIndex):
    _ts_col = next((c for c in ['timestamp', 'ts'] if c in feat.columns), None)
    if _ts_col:
        feat = feat.set_index(_ts_col)
        feat.index = pd.to_datetime(feat.index)
        feat = feat.sort_index()

feed_col = 'feed_total' if 'feed_total' in feat.columns else None
if feed_col is None:
    raise ValueError('feed_total not found — cannot detect runs.')

mask     = cracking_mask(feat[feed_col])
runs_all = segment_runs(mask)

DATA_START = pd.Timestamp(START)
DATA_END   = pd.Timestamp(END)

runs = [r for r in runs_all
        if (r.start - DATA_START) >= pd.Timedelta(hours=MIN_GAP_H)
        and (DATA_END - r.end)   >= pd.Timedelta(hours=MIN_GAP_H)]

print(f'All runs detected       : {len(runs_all)}')
print(f'Complete cycles (kept)  : {len(runs)}')
print(f'Incomplete cycles (cut) : {len(runs_all) - len(runs)}')
print()
for r in runs:
    print(f'  Run {r.index:2d}: {r.start.date()} → {r.end.date()}  ({r.length_days:.1f} d)')

## 5. analysis_window_v2 — 3-layer filter
### Layer 1 — Ground truth (`decoke_air`)
```
decoke_mask = any(decoke_air_A, B, C, D) > 0
cracking_gt = ~decoke_mask
```
### Layer 2 — Setpoint stability (`cot_ctrl`)
```
cot_stable = (|Δcot_ctrl| < 0.5°C) AND (rolling_σ(cot_ctrl, 4h) < 1.0°C)
```
Fallback to measured COT (σ < 2°C / 4h) if `cot_ctrl` is not in the feature matrix.
### Layer 3 — PELT change-point detection
Fits a Pruned Exact Linear Time (PELT) model on `[feed_total, cot_ctrl]` to find
the exact transition from warmup to steady-state (`τ₁`) and from steady-state to decoke (`τ₂`).
Falls back to 12h heuristic if PELT finds no interior breakpoints.
### Combined
```
window_mask = index.between(τ₁, τ₂) AND cracking_gt AND cot_stable
```

In [0]:
import ruptures as rpt


def analysis_window_v2(run, feat_df: pd.DataFrame) -> dict:
    """
    3-layer analysis window detection.
    Returns a dict compatible with 00b's find_analysis_window() output,
    plus extra diagnostic columns: pelt_used, warmup_method, decoke_method.
    """
    seg = feat_df.loc[run.start:run.end].copy()

    null = dict(
        run=run.index, run_start=run.start, run_end=run.end,
        analysis_start=pd.NaT, analysis_end=pd.NaT,
        warmup_hours=np.nan, tail_hours=np.nan,
        length_days=round(run.length_days, 2), clean_days=np.nan,
        settled=False, pelt_used=False,
        warmup_method='none', decoke_method='none',
    )

    if len(seg) < PELT_MIN_SIZE * 2:
        return null

    # ── Layer 1: decoke_air ground truth ────────────────────────────────
    dc_cols = [c for c in seg.columns if c.startswith('decoke_air')]
    if dc_cols:
        decoke_mask  = seg[dc_cols].gt(DECOKE_AIR_THRESH).any(axis=1)
        seg['_cracking_gt'] = ~decoke_mask
        decoke_method = 'decoke_air'
    else:
        # Fallback: feed-based decoke mask (same as 00b)
        run_med = seg['feed_total'][seg['feed_total'] > seg['feed_total'].max() * 0.5].median()
        seg['_cracking_gt'] = seg['feed_total'] > FEED_SETTLED_FRAC * run_med
        decoke_method = 'feed_heuristic'

    # ── Layer 2: cot_ctrl setpoint stability ────────────────────────────
    if 'cot_ctrl' in seg.columns:
        step_change   = seg['cot_ctrl'].diff().abs().fillna(0) > COT_STEP_THRESH
        rolling_std   = seg['cot_ctrl'].rolling(COT_ROLL_STEPS, min_periods=2).std().fillna(999)
        seg['_cot_stable'] = (~step_change) & (rolling_std < COT_STD_ADV_C)
        cot_source    = 'cot_ctrl'
    else:
        # Fallback: measured COT
        _cot = next((c for c in ['cot'] if c in seg.columns), None)
        if _cot:
            rolling_std    = seg[_cot].rolling(COT_ROLL_STEPS, min_periods=2).std().fillna(999)
            seg['_cot_stable'] = rolling_std < COT_STD_THRESH_C
            cot_source     = 'cot_measured_fallback'
        else:
            seg['_cot_stable'] = True
            cot_source     = 'none'

    # ── Layer 3: PELT change-point detection ────────────────────────────
    _signal_cols = [c for c in ['feed_total', 'cot_ctrl', 'cot'] if c in seg.columns][:2]
    pelt_used    = False
    warmup_end   = run.start + pd.Timedelta(hours=MIN_WARMUP_H)  # fallback
    decoke_start = run.end                                        # fallback

    if len(_signal_cols) >= 1 and len(seg) >= PELT_MIN_SIZE * 3:
        try:
            _sig = seg[_signal_cols].ffill().bfill().values
            # z-score normalise so both columns contribute equally
            _mu  = _sig.mean(axis=0)
            _sd  = _sig.std(axis=0) + 1e-9
            _sig = (_sig - _mu) / _sd

            _model = rpt.Pelt(model="rbf", min_size=PELT_MIN_SIZE, jump=1).fit(_sig)
            _bkps  = _model.predict(pen=PELT_PENALTY)
            # _bkps is a list of 1-based indices; last entry = len(seg) (sentinel)
            _interior = [b for b in _bkps if 0 < b < len(seg)]

            if _interior:
                _t1_idx  = _interior[0]
                _t2_idx  = _interior[-1] if len(_interior) > 1 else len(seg) - 1
                warmup_end   = seg.index[min(_t1_idx,   len(seg) - 1)]
                decoke_start = seg.index[min(_t2_idx,   len(seg) - 1)]
                pelt_used    = True

        except Exception as _pelt_err:
            print(f'  PELT failed for run {run.index}: {_pelt_err} — using heuristic fallback')

    warmup_method = 'pelt' if pelt_used else 'heuristic_12h'

    # Enforce minimum warmup regardless of PELT result
    warmup_end = max(warmup_end, run.start + pd.Timedelta(hours=MIN_WARMUP_H))

    # ── Combine all three layers ─────────────────────────────────────────
    window_mask = (
        (seg.index >= warmup_end) & (seg.index <= decoke_start)
        & seg['_cracking_gt']
        & seg['_cot_stable']
    )
    valid_idx = seg.index[window_mask]

    if valid_idx.empty:
        null.update(pelt_used=pelt_used, warmup_method=warmup_method,
                    decoke_method=decoke_method)
        return null

    analysis_start = valid_idx[0]
    analysis_end   = valid_idx[-1]

    return dict(
        run          = run.index,
        run_start    = run.start,
        run_end      = run.end,
        analysis_start = analysis_start,
        analysis_end   = analysis_end,
        warmup_hours   = round((analysis_start - run.start).total_seconds() / 3600, 1),
        tail_hours     = round((run.end - analysis_end).total_seconds() / 3600, 1),
        length_days    = round(run.length_days, 2),
        clean_days     = round((analysis_end - analysis_start).total_seconds() / 86400, 2),
        settled        = True,
        pelt_used      = pelt_used,
        warmup_method  = warmup_method,
        decoke_method  = decoke_method,
    )


windows = [analysis_window_v2(r, feat) for r in runs]
win_df  = pd.DataFrame(windows)

print(f'\nWindows computed for {len(win_df)} runs')
print(f'  Settled          : {win_df["settled"].sum()} / {len(win_df)}')
print(f'  PELT used        : {win_df["pelt_used"].sum()} / {len(win_df)}  '
      f'(rest used 12h heuristic)')
print(f'  Decoke via air   : {(win_df["decoke_method"] == "decoke_air").sum()} / {len(win_df)}')
print(f'  Warmup avg       : {win_df["warmup_hours"].mean():.1f} h  '
      f'(range {win_df["warmup_hours"].min():.0f}–{win_df["warmup_hours"].max():.0f} h)')
print(f'  Tail avg         : {win_df["tail_hours"].mean():.1f} h')
print(f'  Clean avg        : {win_df["clean_days"].mean():.1f} d')
display(win_df[['run','run_start','run_end','analysis_start','analysis_end',
               'warmup_hours','tail_hours','clean_days','settled',
               'pelt_used','warmup_method','decoke_method']])

## 6. Compare 00b (heuristic) vs 00c (v2) windows
Load 00b run windows CSV (if it exists) and show side-by-side delta per run.
Positive `Δclean_days` means 00c found a wider valid window.

In [0]:
win_00b_path = os.path.join(REPO_ROOT, 'output', FURNACE, '00b_clean', 'run_analysis_windows.csv')

if os.path.exists(win_00b_path):
    win_00b = pd.read_csv(win_00b_path,
                          parse_dates=['run_start','run_end','analysis_start','analysis_end'])
    # Merge on run index
    cmp = win_df[['run','warmup_hours','tail_hours','clean_days','pelt_used','decoke_method']].copy()
    cmp = cmp.rename(columns={
        'warmup_hours': 'warmup_h_v2',
        'tail_hours':   'tail_h_v2',
        'clean_days':   'clean_d_v2',
    })
    win_00b_sub = win_00b[['run','warmup_hours','tail_hours','clean_days']].rename(columns={
        'warmup_hours': 'warmup_h_v1',
        'tail_hours':   'tail_h_v1',
        'clean_days':   'clean_d_v1',
    })
    cmp = cmp.merge(win_00b_sub, on='run', how='left')
    cmp['Δclean_days'] = (cmp['clean_d_v2'] - cmp['clean_d_v1']).round(2)
    cmp['Δwarmup_h']   = (cmp['warmup_h_v2'] - cmp['warmup_h_v1']).round(1)

    print('Comparison: 00b heuristic vs 00c v2')
    print(f'  Mean Δclean_days : {cmp["Δclean_days"].mean():+.2f} d  '
          f'(positive = 00c found more clean data)')
    print(f'  Mean Δwarmup_h   : {cmp["Δwarmup_h"].mean():+.1f} h  '
          f'(negative = shorter warmup exclusion)')
    display(cmp[['run','clean_d_v1','clean_d_v2','Δclean_days',
                  'warmup_h_v1','warmup_h_v2','Δwarmup_h',
                  'pelt_used','decoke_method']])
    comparison_available = True
else:
    print(f'00b windows CSV not found at {win_00b_path}')
    print('Run notebook_00b_clean_extraction.ipynb first to enable comparison.')
    comparison_available = False
    win_00b = None

## 7. Apply all cuts — build clean time series

In [0]:
clean_mask = pd.Series(False, index=feat.index)
run_id_col = pd.Series(np.nan,  index=feat.index)

valid_runs = win_df[win_df['analysis_start'].notna() & win_df['analysis_end'].notna()].copy()

for _, w in valid_runs.iterrows():
    a_start = w['analysis_start']
    a_end   = w['analysis_end']
    if pd.isna(a_start) or pd.isna(a_end) or a_end <= a_start:
        continue
    in_window = (feat.index >= a_start) & (feat.index <= a_end)
    clean_mask[in_window] = True
    run_id_col[in_window] = int(w['run'])

feat_clean = feat[clean_mask].copy()
feat_clean['run_id'] = run_id_col[clean_mask].astype(int)
feat_clean['phase']  = 'clean_adv'

total_rows = len(feat)
clean_rows = len(feat_clean)
pct_kept   = 100 * clean_rows / total_rows

print(f'Total rows           : {total_rows:,}')
print(f'Rows kept (v2)       : {clean_rows:,}  ({pct_kept:.1f}%)')
print(f'Rows removed         : {total_rows - clean_rows:,}  ({100-pct_kept:.1f}%)')
print(f'Complete runs        : {feat_clean["run_id"].nunique()}')

## 8. Manual exclusions
Same as 00b — add known sensor faults after DCS verification.

In [0]:
MANUAL_EXCLUDE = [
    # ('Run9_sensor_fault', '2025-10-28', '2025-11-02'),
]

excluded_count = 0
for reason, t0, t1 in MANUAL_EXCLUDE:
    mask_exc = (feat_clean.index >= t0) & (feat_clean.index <= t1)
    n = mask_exc.sum()
    if 'dd_abs_max' in feat_clean.columns:
        feat_clean.loc[mask_exc, 'dd_abs_max'] = np.nan
    excluded_count += n
    print(f'  Manual exclusion: {n} rows NaN-ed — {reason}')

if not MANUAL_EXCLUDE:
    print('No manual exclusions applied.')

## 9. Save clean data to Databricks Delta table
New table name: `{FURNACE}_clean_adv` — keeps 00b output intact for comparison.

In [0]:
DELTA_TABLE_CLEAN_ADV = f'{DELTA_CATALOG}.{FURNACE.lower()}_clean_adv'

sdf_clean = spark.createDataFrame(
    feat_clean.reset_index().rename(columns={'index': 'timestamp'})
)
(sdf_clean.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(DELTA_TABLE_CLEAN_ADV))

print(f'Clean data (v2) saved to Delta : {DELTA_TABLE_CLEAN_ADV}')
print(f'Shape                          : {feat_clean.shape}')

win_path_adv = os.path.join(OUTPUT_DIR, 'run_analysis_windows_adv.csv')
win_df.to_csv(win_path_adv, index=False)
print(f'Run windows metadata saved     : {win_path_adv}')

## 10. Verification plot
5-panel plot: Feed, COT setpoint, decoke_air sum, dd_abs_max, and (if available) v1 vs v2 warmup boundary overlay.

In [0]:
n_panels  = 5 if decoke_air_cols else 4
fig, axes = plt.subplots(n_panels, 1, figsize=(16, 3*n_panels), sharex=True)
fig.suptitle(
    f'{FURNACE} — 00c Advanced clean extraction  '
    f'(green=kept · red=warmup · orange=tail · purple=decoke)',
    fontsize=11, fontweight='bold'
)

ax_feed, ax_cot, ax_dc, ax_dd = axes[0], axes[1], axes[2], axes[3]
ax_pass = axes[4] if n_panels == 5 else None

# Panel 1: Feed
if 'feed_total' in feat.columns:
    ax_feed.plot(feat.index,       feat['feed_total'],       color='#378add', lw=0.5, alpha=0.5, label='raw')
    ax_feed.plot(feat_clean.index, feat_clean['feed_total'], color='#1d9e75', lw=0.9, label='clean (v2)')
ax_feed.set_ylabel('Feed (NM³/H)')
ax_feed.legend(fontsize=8)

# Panel 2: COT / cot_ctrl
cot_col_plot = 'cot_ctrl' if 'cot_ctrl' in feat.columns else next(
    (c for c in ['cot'] if c in feat.columns), None)
if cot_col_plot:
    ax_cot.plot(feat.index,       feat[cot_col_plot],       color='#7f77dd', lw=0.5, alpha=0.5, label='raw')
    ax_cot.plot(feat_clean.index, feat_clean[cot_col_plot], color='#085041', lw=0.9,
                label=f'{cot_col_plot} (clean v2)')
ax_cot.set_ylabel('COT / setpoint (°C)')
ax_cot.legend(fontsize=8)

# Panel 3: decoke_air sum
if decoke_air_cols:
    decoke_sum = feat[decoke_air_cols].sum(axis=1)
    ax_dc.fill_between(feat.index, decoke_sum, alpha=0.6, color='#e24b4a', label='decoke_air sum')
    ax_dc.axhline(0, color='#e24b4a', lw=0.5)
    ax_dc.set_ylabel('Σ decoke_air\n(Nm³/H)')
    ax_dc.legend(fontsize=8)
else:
    ax_dc.text(0.5, 0.5, 'decoke_air not in feature matrix — Layer 1 used feed heuristic',
               ha='center', va='center', transform=ax_dc.transAxes, color='grey')
    ax_dc.set_ylabel('decoke_air')

# Panel 4: dd_abs_max
if 'dd_abs_max' in feat.columns:
    ax_dd.plot(feat.index,       feat['dd_abs_max'],       color='#ef9f27', lw=0.5, alpha=0.4, label='raw')
    ax_dd.plot(feat_clean.index, feat_clean['dd_abs_max'], color='#d85a30', lw=0.9,  label='clean (v2)')
    ax_dd.axhline(45, color='#e24b4a', ls='--', lw=1.0, label='Alarm 45°C')
    ax_dd.axhline(30, color='#ef9f27', ls='--', lw=1.0, label='Pre-alarm 30°C')
ax_dd.set_ylabel('dd_abs_max (°C)')
ax_dd.legend(fontsize=8)

# Panel 5: per-pass dd
if ax_pass is not None:
    pass_colors = {'A': '#1f77b4', 'B': '#ff7f0e', 'C': '#2ca02c', 'D': '#d62728'}
    for p, pc in pass_colors.items():
        col = f'dd_abs_max_{p}'
        if col in feat_clean.columns:
            ax_pass.plot(feat_clean.index, feat_clean[col], lw=0.7, color=pc, label=f'Pass {p}')
    ax_pass.axhline(45, color='#e24b4a', ls='--', lw=0.8)
    ax_pass.set_ylabel('dd per pass (°C)')
    ax_pass.legend(fontsize=8, ncol=2)

# Shade windows — v2 (main) + optional v1 boundary dashes for comparison
for _, w in valid_runs.iterrows():
    for ax in axes:
        ax.axvspan(w['run_start'],      w['analysis_start'], alpha=0.12, color='#e24b4a', zorder=0)
        ax.axvspan(w['analysis_start'], w['analysis_end'],   alpha=0.08, color='#1d9e75', zorder=0)
        ax.axvspan(w['analysis_end'],   w['run_end'],        alpha=0.12, color='#ef9f27', zorder=0)

# Overlay 00b boundaries as dashed vertical lines (if available)
if comparison_available and win_00b is not None:
    for _, w1 in win_00b.iterrows():
        if pd.notna(w1['analysis_start']):
            for ax in axes:
                ax.axvline(w1['analysis_start'], color='grey', lw=0.8, ls=':', alpha=0.7,
                           label='00b warmup_end' if _ == 0 else '')
                ax.axvline(w1['analysis_end'],   color='grey', lw=0.8, ls=':', alpha=0.7)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, f'{FURNACE}_00c_clean_extraction_adv.png')
plt.savefig(plot_path, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')

## 11. Summary

In [0]:
print('=' * 65)
print(f'ADVANCED CLEAN EXTRACTION SUMMARY — {FURNACE}')
print('=' * 65)
print(f'  Furnace          : {FURNACE}')
print(f'  Raw data window  : {START} → {END}')
print(f'  Total runs found : {len(runs)}')
print(f'  Complete cycles  : {len(valid_runs)}')
print(f'  Clean rows kept  : {clean_rows:,}  ({pct_kept:.1f}% of raw)')
print(f'  Total clean time : {clean_rows * BUCKET_MIN / 60 / 24:.1f} days')
print()
print('Layer usage:')
print(f'  Layer 1 (decoke_air)   : {(win_df["decoke_method"]=="decoke_air").sum()} / {len(win_df)} runs used ground truth')
print(f'  Layer 2 (cot_ctrl)     : {"cot_ctrl" in feat.columns}  (setpoint available in feature matrix)')
print(f'  Layer 3 (PELT)         : {win_df["pelt_used"].sum()} / {len(win_df)} runs used change-point detection')
print()
print('Data storage (all in Databricks):')
print(f'  Feature matrix   : {DELTA_TABLE_FEAT}')
print(f'  Clean data (v2)  : {DELTA_TABLE_CLEAN_ADV}')
print(f'  Run windows CSV  : {win_path_adv}')
print(f'  Verification PNG : {plot_path}')
print()
print('How to use in downstream notebooks:')
print(f'  feat_clean = spark.table("{DELTA_TABLE_CLEAN_ADV}").toPandas()')
print( '  feat_clean = feat_clean.set_index("timestamp")')
print( '  # phase column = "clean_adv"  |  run_id column labels each run')
print()

suspicious = win_df[(win_df['warmup_hours'] > 48) | (win_df['tail_hours'] > 48)]
if len(suspicious):
    print('⚠  Suspicious runs (warmup or tail > 48h) — verify with DCS:')
    for _, s in suspicious.iterrows():
        print(f'   Run {int(s["run"]):2d}: warmup={s["warmup_hours"]:.0f}h, '
              f'tail={s["tail_hours"]:.0f}h  method={s["warmup_method"]}')

## 12. Standalone reload section
Run this section independently (after the notebook has been executed once) to reload
existing outputs without rerunning the full pipeline.

In [0]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT  = os.path.dirname(os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, REPO_ROOT)
os.environ['DDF_CLUSTER_ID'] = '0612-154044-62gqpkte'

from olefins_ddf.io_events import get_spark

FURNACE       = '1HA'
DELTA_CATALOG = 'indorama_corporate_olefins_paas_azure_weu_dev_research.workspaces'
OUTPUT_DIR    = os.path.join(REPO_ROOT, 'output', FURNACE, '00c_clean_adv')
BUCKET_MIN    = 30

spark = get_spark()

feat = spark.table(f'{DELTA_CATALOG}.{FURNACE.lower()}_features').toPandas()
ts_col = next(c for c in ['timestamp','ts'] if c in feat.columns)
feat = feat.set_index(ts_col)
feat.index = pd.to_datetime(feat.index)
feat = feat.sort_index()
feat.index.name = 'timestamp'

feat_clean = spark.table(f'{DELTA_CATALOG}.{FURNACE.lower()}_clean_adv').toPandas()
ts_col = next(c for c in ['timestamp','ts'] if c in feat_clean.columns)
feat_clean = feat_clean.set_index(ts_col)
feat_clean.index = pd.to_datetime(feat_clean.index)
feat_clean = feat_clean.sort_index()
feat_clean.index.name = 'timestamp'

win_path_adv = os.path.join(OUTPUT_DIR, 'run_analysis_windows_adv.csv')
valid_runs   = pd.read_csv(win_path_adv,
                parse_dates=['run_start','run_end','analysis_start','analysis_end'])

print(f'feat      : {feat.shape}')
print(f'feat_clean: {feat_clean.shape}')
print(f'runs      : {len(valid_runs)}')

## 13. Per-cycle plots (run-age aligned)

In [0]:
CYCLE_DIR   = os.path.join(OUTPUT_DIR, 'cycles')
os.makedirs(CYCLE_DIR, exist_ok=True)

PASS_COLORS = {'A': '#1f77b4', 'B': '#ff7f0e', 'C': '#2ca02c', 'D': '#d62728'}
X_MAX       = valid_runs['length_days'].max() + 1.0
cot_col     = next((c for c in ['cot_ctrl', 'cot'] if c in feat_clean.columns), None)

for _, w in valid_runs.iterrows():
    rid   = int(w['run'])
    seg   = feat_clean[feat_clean['run_id'] == rid].copy()

    if seg.empty or 'run_age_days' not in seg.columns:
        print(f'Run {rid}: skipped (no data or missing run_age_days)')
        continue

    x        = seg['run_age_days']
    warmup_d = w['warmup_hours'] / 24
    tail_d   = w['tail_hours'] / 24
    end_d    = w['length_days']
    flag     = ' ⚠ VERIFY DCS' if (w['warmup_hours'] > 48 or w['tail_hours'] > 48) else ''
    pelt_str = '  [PELT]' if w.get('pelt_used', False) else '  [12h heuristic]'
    dc_str   = '  [decoke_air GT]' if w.get('decoke_method') == 'decoke_air' else '  [feed heuristic]'

    fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)
    fig.suptitle(
        f'{FURNACE}  Run {rid}  |  {str(w["run_start"])[:10]} → {str(w["run_end"])[:10]}'
        f'  ({end_d:.1f}d total · {w["clean_days"]:.1f}d clean'
        f' · warmup {w["warmup_hours"]:.0f}h · tail {w["tail_hours"]:.0f}h)'
        f'{pelt_str}{dc_str}{flag}',
        fontsize=9, fontweight='bold',
        color='#c0392b' if flag else 'black'
    )

    ax1, ax2, ax3, ax4 = axes

    if 'feed_total' in seg.columns:
        ax1.plot(x, seg['feed_total'], color='teal', lw=0.8)
    ax1.set_ylabel('Feed (NM³/H)')

    if cot_col and cot_col in seg.columns:
        ax2.plot(x, seg[cot_col], color='darkgreen', lw=0.8)
    ax2.set_ylabel('COT / setpoint (°C)')

    if 'dd_abs_max' in seg.columns:
        ax3.plot(x, seg['dd_abs_max'], color='tomato', lw=0.9)
    ax3.axhline(45, color='red',    ls='--', lw=1.0, label='Alarm 45°C')
    ax3.axhline(30, color='orange', ls='--', lw=1.0, label='Pre-alarm 30°C')
    ax3.set_ylabel('dd_abs_max (°C)')
    ax3.legend(fontsize=7, loc='upper right')

    for p, pc in PASS_COLORS.items():
        col_name = f'dd_abs_max_{p}'
        if col_name in seg.columns:
            ax4.plot(x, seg[col_name], lw=0.7, color=pc, label=f'Pass {p}')
    ax4.axhline(45, color='red',    ls='--', lw=1.0)
    ax4.axhline(30, color='orange', ls='--', lw=1.0)
    ax4.set_ylabel('dd per pass (°C)')
    ax4.set_xlabel('Run age (days)')
    ax4.legend(fontsize=7, loc='upper right', ncol=2)

    for ax in axes:
        ax.set_xlim(0, X_MAX)
        ax.axvspan(0,        warmup_d,           alpha=0.10, color='grey')
        ax.axvline(warmup_d, color='grey',   lw=0.8, ls='--')
        if tail_d > 0:
            ax.axvspan(end_d - tail_d, end_d, alpha=0.10, color='orange')
            ax.axvline(end_d - tail_d, color='orange', lw=0.8, ls='--')

    plt.tight_layout()
    save_path = os.path.join(CYCLE_DIR, f'{FURNACE}_run{rid:02d}_adv.png')
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'  Run {rid:2d} → {save_path}')

print(f'\nDone — {len(valid_runs)} cycle plots saved to {CYCLE_DIR}')